# EDA

In [10]:
import os
import pandas as pd
import csv

data_dir = '../data/' if os.path.exists('../data/') else 'data/'
files = ['people.csv', 'org_units.csv', 'seats.csv', 'price_list.csv', 'activity_events.csv']
dfs = {}

for f in files:
    path = os.path.join(data_dir, f)
    if os.path.exists(path):
        df = pd.read_csv(path)
        dfs[f] = df
        print(f"=== {f} ===")
        print(f"Shape: {df.shape}")
        print("Null counts:")
        print(df.isna().sum())
        print("Head:")
        print(df.head(3))
        print("\n" + "-"*40 + "\n")

=== people.csv ===
Shape: (400, 5)
Null counts:
user_id        0
name           0
org_unit_id    0
role           0
location       0
dtype: int64
Head:
  user_id            name org_unit_id              role   location
0   U0001  Aarav Banerjee       ORG04  Senior Developer  Bengaluru
1   U0002   Diya Banerjee       ORG05         Developer       Pune
2   U0003   Taylor Kapoor       ORG06  Senior Developer       Pune

----------------------------------------

=== org_units.csv ===
Shape: (37, 4)
Null counts:
org_unit_id    0
name           0
parent_id      1
level          0
dtype: int64
Head:
  org_unit_id              name parent_id level
0       ORG01       Engineering       NaN    L3
1       ORG02          Platform     ORG01    L4
2       ORG03  Platform Group 1     ORG02    L5

----------------------------------------

=== seats.csv ===
Shape: (680, 5)
Null counts:
user_id         0
tool            0
tier            0
granted_on      0
revoked_on    605
dtype: int64
Head:
  user_id

In [11]:
seats_df = dfs.get('seats.csv')
blank_revoked = seats_df[seats_df['revoked_on'].isna()]
print(f"Count of blank revoked_on: {len(blank_revoked)}")
print(blank_revoked.head())

Count of blank revoked_on: 605
  user_id tool      tier  granted_on revoked_on
0   U0001    C  standard  2026-05-28        NaN
1   U0002    D  standard  2026-03-25        NaN
2   U0002    E  standard  2026-02-01        NaN
3   U0003    A  standard  2026-03-03        NaN
4   U0003    E  standard  2026-03-25        NaN


In [12]:
for f in files:
    path = os.path.join(data_dir, f)
    if os.path.exists(path):
        with open(path, 'rb') as file:
            content = file.read(1000)
            has_crlf = b'\r\n' in content
            print(f"{f} uses CRLF: {has_crlf}")

print("\n--- Naive split vs CSV parser ---")
sample_line = "u1,tool_a,pro,2026-01-01,\r\n"
print("Naive string.split(','):", sample_line.split(','))
reader = csv.reader([sample_line])
print("csv.reader parse:", next(reader))

people.csv uses CRLF: True
org_units.csv uses CRLF: True
seats.csv uses CRLF: True
price_list.csv uses CRLF: True
activity_events.csv uses CRLF: True

--- Naive split vs CSV parser ---
Naive string.split(','): ['u1', 'tool_a', 'pro', '2026-01-01', '\r\n']
csv.reader parse: ['u1', 'tool_a', 'pro', '2026-01-01', '']


In [13]:
people_df = dfs.get('people.csv')
people_ids = set(people_df['user_id'].dropna())
seat_ids = set(seats_df['user_id'].dropna())

orphan_seats = seat_ids - people_ids
print(f"Orphan IDs in seats: {len(orphan_seats)}")
if orphan_seats:
    print(f"Sample: {list(orphan_seats)[:5]}")

Orphan IDs in seats: 0


In [14]:
events_df = dfs.get('activity_events.csv')
event_ids = set(events_df['user_id'].dropna())
orphan_events = event_ids - people_ids

print(f"Orphan IDs in events: {len(orphan_events)}")
if orphan_events:
    print(f"Sample: {list(orphan_events)[:5]}")

Orphan IDs in events: 5
Sample: ['U9001', 'U9003', 'U9005', 'U9002', 'U9004']


In [15]:
seat_pairs = set(zip(seats_df['user_id'], seats_df['tool']))
event_pairs = set(zip(events_df['user_id'], events_df['tool']))

unlicensed_usage = event_pairs - seat_pairs
print(f"Usage events with no corresponding seat ever granted: {len(unlicensed_usage)} pairs")
if unlicensed_usage:
    print(f"Sample: {list(unlicensed_usage)[:5]}")

Usage events with no corresponding seat ever granted: 135 pairs
Sample: [('U0332', 'D'), ('U0020', 'E'), ('U0360', 'B'), ('U0162', 'A'), ('U0047', 'D')]


In [16]:
s_df = seats_df.copy()
s_df['granted_on'] = pd.to_datetime(s_df['granted_on'])
s_df['revoked_on'] = pd.to_datetime(s_df['revoked_on'])

bad_dates = s_df[s_df['granted_on'] > s_df['revoked_on']]
print(f"Seats with granted_on > revoked_on: {len(bad_dates)}")

Seats with granted_on > revoked_on: 0


In [17]:
prices_df = dfs.get('price_list.csv')
tool_counts = prices_df.groupby('tool').size()
multi_tier_tools = tool_counts[tool_counts > 1].index

print(f"Tools with multiple tiers: {len(multi_tier_tools)}")
print(prices_df[prices_df['tool'].isin(multi_tier_tools)].sort_values(['tool', 'tier']))

Tools with multiple tiers: 2
  tool        tier  monthly_price_usd
1    A         pro               39.0
0    A    standard               19.0
4    C  enterprise               60.0
3    C    standard               30.0


In [18]:
unpriced = prices_df[prices_df['monthly_price_usd'].isna()]
print(f"Unpriced rows: {len(unpriced)}")
print(unpriced[['tool', 'tier']])

Unpriced rows: 1
  tool      tier
6    E  standard


In [19]:
seat_groups = seats_df.groupby(['user_id', 'tool']).size()
duplicates = seat_groups[seat_groups > 1].index

print(f"(user_id, tool) pairs with multiple seat rows: {len(duplicates)}")
if len(duplicates) > 0:
    sample_dupes = seats_df[seats_df.set_index(['user_id', 'tool']).index.isin(duplicates[:2])]
    print(sample_dupes.sort_values(['user_id', 'tool', 'granted_on']))

(user_id, tool) pairs with multiple seat rows: 0


In [20]:
overlapping_pairs = []
s_df['end'] = s_df['revoked_on'].fillna(pd.Timestamp('2099-12-31'))
dupe_df = s_df[s_df.set_index(['user_id', 'tool']).index.isin(duplicates)]

for (u, t), group in dupe_df.groupby(['user_id', 'tool']):
    group = group.sort_values('granted_on')
    prev_end = None
    for _, row in group.iterrows():
        if prev_end is not None and row['granted_on'] <= prev_end:
            overlap_days = (min(row['end'], prev_end) - row['granted_on']).days + 1
            if overlap_days > 0:
                overlapping_pairs.append({'user_id': u, 'tool': t, 'overlap_days': overlap_days})
        
        if prev_end is None or row['end'] > prev_end:
            prev_end = row['end']

print(f"Found {len(overlapping_pairs)} pairs with explicitly overlapping date ranges.")
if overlapping_pairs:
    print(pd.DataFrame(overlapping_pairs).head())

Found 0 pairs with explicitly overlapping date ranges.


Based on this pass, the ingestion and metrics layer needs to handle a few specific realities:
1. **Line endings:** The files use CRLF. A standard CSV parser handles this, but a manual string split would break.
2. **Ghost users:** There are IDs in the seats and events files that don't exist in `people.csv`. They have to be filtered out.
3. **Rogue usage:** People are logging events for tools they don't have seats for. Just joining events isn't enough; we have to verify seat possession on the exact event date.
4. **Composite pricing:** Price lookups need both `tool` and `tier`.
5. **Unpriced tools:** Some prices are genuinely blank and need to be handled gracefully (e.g., returning nulls, not 0s).
6. **Open-ended seats:** A blank `revoked_on` means the seat is still active today.
7. **Multiple seats:** Users can hold multiple seat rows for the same tool. To calculate licensed users and spend correctly, we have to union their held days to avoid double counting overlaps.